In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import scrublet as scr
import seaborn as sns
import scipy.stats
import anndata
import os

In [2]:
adata = sc.read_h5ad("./RNA_merged_postQC.h5ad")
adata

AnnData object with n_obs × n_vars = 180801 × 36601
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt'
    var: 'features'

In [3]:
def bh(pvalues):
    '''
    Computes the Benjamini-Hochberg FDR correction.

    Input:
        * pvals - vector of p-values to correct
    '''
    n = int(pvalues.shape[0])
    new_pvalues = np.empty(n)
    values = [ (pvalue, i) for i, pvalue in enumerate(pvalues) ]
    values.sort()
    values.reverse()
    new_values = []
    for i, vals in enumerate(values):
        rank = n - i
        pvalue, index = vals
        new_values.append((n/rank) * pvalue)
    for i in range(0, int(n)-1):
        if new_values[i] < new_values[i+1]:
            new_values[i+1] = new_values[i]
    for i, vals in enumerate(values):
        pvalue, index = vals
        new_pvalues[index] = new_values[i]
    return new_pvalues

In [4]:
### Change directory 
    
CWD = "/Users/jamrute/Desktop/"

from os import chdir
chdir(CWD)

In [5]:
len(adata.obs['sample'].unique())

45

In [6]:
####### main
meta = adata.obs
sc.settings.verbosity = 1  # verbosity: errors (0), warnings (1), info (2), hints (3)

scorenames = ['scrublet_score','scrublet_cluster_score','bh_pval']
os.makedirs('scrublet-scores')

###

for sample in meta['sample'].unique():
    print(sample)
    #import data
    adata_sample = adata[adata.obs['sample'] == sample]
    
    #set up and run Scrublet
    scrub = scr.Scrublet(adata_sample.X)

    doublet_scores, predicted_doublets = scrub.scrub_doublets(verbose=False)
    
    adata_sample.obs['scrublet_score'] = doublet_scores
    #overcluster prep. run turbo basic scanpy pipeline
    sc.pp.filter_genes(adata_sample, min_cells=3)
    sc.pp.normalize_per_cell(adata_sample, counts_per_cell_after=1e4)
    sc.pp.log1p(adata_sample)
    sc.pp.highly_variable_genes(adata_sample, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata_sample = adata_sample[:, adata_sample.var['highly_variable']]
    sc.pp.scale(adata_sample, max_value=10)
    sc.tl.pca(adata_sample, svd_solver='arpack')
    sc.pp.neighbors(adata_sample)
    #eoverclustering proper - do basic clustering first, then cluster each cluster
    sc.tl.louvain(adata_sample)
    for clus in np.unique(adata_sample.obs['louvain']):
        sc.tl.louvain(adata_sample, restrict_to=('louvain',[clus]))
        adata_sample.obs['louvain'] = adata_sample.obs['louvain_R']
    #compute the cluster scores - the median of Scrublet scores per overclustered cluster
    for clus in np.unique(adata_sample.obs['louvain']):
        adata_sample.obs.loc[adata_sample.obs['louvain']==clus, 'scrublet_cluster_score'] = \
            np.median(adata_sample.obs.loc[adata_sample.obs['louvain']==clus, 'scrublet_score'])
    #now compute doublet p-values. figure out the median and mad (from above-median values) for the distribution
    med = np.median(adata_sample.obs['scrublet_cluster_score'])
    mask = adata_sample.obs['scrublet_cluster_score']>med
    mad = np.median(adata_sample.obs['scrublet_cluster_score'][mask]-med)
    #let's do a one-sided test. the Bertie write-up does not address this but it makes sense
    pvals = 1-scipy.stats.norm.cdf(adata_sample.obs['scrublet_cluster_score'], loc=med, scale=1.4826*mad)
    adata_sample.obs['bh_pval'] = bh(pvals)
    #create results data frame for single sample and copy stuff over from the adata object
    scrublet_sample = pd.DataFrame(0, index=adata_sample.obs_names, columns=scorenames)
    for meta in scorenames:
        scrublet_sample[meta] = adata_sample.obs[meta]
    #write out complete sample scores
    scrublet_sample.to_csv('scrublet-scores/'+sample+'.csv')

MGI2013_MGI2036_TWAP-T1182


/var/folders/92/4qr0bhyn67g8htdht23h146h0000gn/T/ipykernel_2353/1137114659.py:20: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['scrublet_score'] = doublet_scores
/Users/jamrute/opt/anaconda3/lib/python3.9/contextlib.py:126: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  next(self.gen)
/Users/jamrute/opt/anaconda3/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


MGI2078_MGI2080_TWAP-T1106


/var/folders/92/4qr0bhyn67g8htdht23h146h0000gn/T/ipykernel_2353/1137114659.py:20: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['scrublet_score'] = doublet_scores
/Users/jamrute/opt/anaconda3/lib/python3.9/contextlib.py:126: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  next(self.gen)
/Users/jamrute/opt/anaconda3/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


MGI2129_MGI2308_MGI2311_TWAP-T1069L


...

/var/folders/92/4qr0bhyn67g8htdht23h146h0000gn/T/ipykernel_2353/1137114659.py:20: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['scrublet_score'] = doublet_scores
/Users/jamrute/opt/anaconda3/lib/python3.9/contextlib.py:126: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  next(self.gen)
/Users/jamrute/opt/anaconda3/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


MGI3339_MGI3341_TWKH-T1214


/var/folders/92/4qr0bhyn67g8htdht23h146h0000gn/T/ipykernel_2353/1137114659.py:20: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['scrublet_score'] = doublet_scores
/Users/jamrute/opt/anaconda3/lib/python3.9/contextlib.py:126: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  next(self.gen)
/Users/jamrute/opt/anaconda3/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


MGI3339_MGI3341_TWKH-T1215


/var/folders/92/4qr0bhyn67g8htdht23h146h0000gn/T/ipykernel_2353/1137114659.py:20: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['scrublet_score'] = doublet_scores
/Users/jamrute/opt/anaconda3/lib/python3.9/contextlib.py:126: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  next(self.gen)
/Users/jamrute/opt/anaconda3/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:843: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [7]:
adata.obs['sample'].unique()

array(['MGI2013_MGI2036_TWAP-T1182', 'MGI2078_MGI2080_TWAP-T1106',
       'MGI2129_MGI2308_MGI2311_TWAP-T1069L',
       'MGI2129_MGI2308_MGI2311_TWAP-T1106R',
       'MGI2131_MGI2132_TWAP-T1114R', 'MGI2578_MGI2581_TWKH-D1127',
       'MGI2578_MGI2581_TWKH-D1144', 'MGI2578_MGI2581_TWKH-D1148',
       'MGI2578_MGI2581_TWKH-T1103', 'MGI2578_MGI2581_TWKH-T1133',
       'MGI2578_MGI2581_TWKH-T1134', 'MGI2578_MGI2581_TWKH-T1135',
       'MGI2578_MGI2581_TWKH-T1137', 'MGI2578_MGI2581_TWKH-T1145',
       'MGI2578_MGI2581_TWKH-T1149', 'MGI2578_MGI2581_TWKH-T1163',
       'MGI2701_MGI2706_TWKH-D1173', 'MGI2701_MGI2706_TWKH-T1096',
       'MGI2701_MGI2706_TWKH-T1139', 'MGI2701_MGI2706_TWKH-T1171',
       'MGI2904_MGI2917_TWKH-JA_T1179', 'MGI2904_MGI2917_TWKH-JA_T1182',
       'MGI2904_MGI2917_TWKH-JA_T1183', 'MGI2904_MGI2917_TWKH-JA_T1184',
       'MGI2904_MGI2917_TWKH-JA_T1185', 'MGI2904_MGI2917_TWKH-JA_T1186',
       'MGI2904_MGI2917_TWKH-JA_T1187', 'MGI2904_MGI2917_TWKH-JA_T1189',
       'MGI3

In [8]:
s1 = pd.read_csv("./scrublet-scores/MGI2013_MGI2036_TWAP-T1182.csv", index_col=0)
s2 = pd.read_csv("./scrublet-scores/MGI2078_MGI2080_TWAP-T1106.csv", index_col=0)
s3 = pd.read_csv("./scrublet-scores/MGI2129_MGI2308_MGI2311_TWAP-T1069L.csv", index_col=0)
s4 = pd.read_csv("./scrublet-scores/MGI2129_MGI2308_MGI2311_TWAP-T1106R.csv", index_col=0)
s5 = pd.read_csv("./scrublet-scores/MGI2131_MGI2132_TWAP-T1114R.csv", index_col=0)
s6 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-D1127.csv", index_col=0)
s7 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-D1144.csv", index_col=0)
s8 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1215.csv", index_col=0)
s9 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-D1148.csv", index_col=0)
s10 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-T1103.csv", index_col=0)
s11 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-T1133.csv", index_col=0)
s12 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-T1134.csv", index_col=0)
s13 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-T1135.csv", index_col=0)
s14 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-T1137.csv", index_col=0)
s15 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-T1145.csv", index_col=0)
s16 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-T1149.csv", index_col=0)
s17 = pd.read_csv("./scrublet-scores/MGI2578_MGI2581_TWKH-T1163.csv", index_col=0)
s18 = pd.read_csv("./scrublet-scores/MGI2701_MGI2706_TWKH-D1173.csv", index_col=0)
s19 = pd.read_csv("./scrublet-scores/MGI2701_MGI2706_TWKH-T1096.csv", index_col=0)
s20 = pd.read_csv("./scrublet-scores/MGI2701_MGI2706_TWKH-T1139.csv", index_col=0)
s21 = pd.read_csv("./scrublet-scores/MGI2701_MGI2706_TWKH-T1171.csv", index_col=0)
s22 = pd.read_csv("./scrublet-scores/MGI2904_MGI2917_TWKH-JA_T1179.csv", index_col=0)
s23 = pd.read_csv("./scrublet-scores/MGI2904_MGI2917_TWKH-JA_T1182.csv", index_col=0)
s24 = pd.read_csv("./scrublet-scores/MGI2904_MGI2917_TWKH-JA_T1183.csv", index_col=0)
s25 = pd.read_csv("./scrublet-scores/MGI2904_MGI2917_TWKH-JA_T1184.csv", index_col=0)
s26 = pd.read_csv("./scrublet-scores/MGI2904_MGI2917_TWKH-JA_T1185.csv", index_col=0)
s27 = pd.read_csv("./scrublet-scores/MGI2904_MGI2917_TWKH-JA_T1186.csv", index_col=0)
s28 = pd.read_csv("./scrublet-scores/MGI2904_MGI2917_TWKH-JA_T1187.csv", index_col=0)
s29 = pd.read_csv("./scrublet-scores/MGI2904_MGI2917_TWKH-JA_T1189.csv", index_col=0)
s30 = pd.read_csv("./scrublet-scores/MGI3172_MGI3298_TWKH-T1036.csv", index_col=0)
s31 = pd.read_csv("./scrublet-scores/MGI3172_MGI3298_TWKH-T1038.csv", index_col=0)
s32 = pd.read_csv("./scrublet-scores/MGI3172_MGI3298_TWKH-T1040.csv", index_col=0)
s33 = pd.read_csv("./scrublet-scores/MGI3172_MGI3298_TWKH-T1090.csv", index_col=0)
s34 = pd.read_csv("./scrublet-scores/MGI3172_MGI3298_TWKH-T1091.csv", index_col=0)
s35 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1022.csv", index_col=0)
s36 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1055.csv", index_col=0)
s37 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1123.csv", index_col=0)
s38 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1127.csv", index_col=0)
s39 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1167.csv", index_col=0)
s40 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1169.csv", index_col=0)
s41 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1190.csv", index_col=0)
s42 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1201.csv", index_col=0)
s43 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1210.csv", index_col=0)
s44 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1211.csv", index_col=0)
s45 = pd.read_csv("./scrublet-scores/MGI3339_MGI3341_TWKH-T1214.csv", index_col=0)

In [9]:
df_list = [s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,s22,s23,s24,
           s25,s26,s27,s28,s29,s30,s31,s32,s33,s34,s35,s36,s37,s38,s39,s40,s41,s42,s43,s44,s45]
scrub_all = pd.concat(df_list)

In [10]:
scrub_all.to_csv('scrublet-scores/all.csv')